# IPL 2026 Matches RAG System using ChromaDB

In [13]:
import os
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from dotenv import load_dotenv
load_dotenv('../.env')

True

### Load the IPL 2026 Matches Dataset

In [14]:
df = pd.read_csv('dataset/ipl_matches_2026.csv')
print(f"Loaded {len(df)} matches.")
df.head(2)

Loaded 74 matches.


,date,season,city,venue,team1,team2,toss_winner,toss_decision,team1_runs,team1_wickets,...,win_by_wickets,player_of_match,match_referee,umpire1,umpire2,tv_umpire,reserve_umpire,overs_limit,team1_players,team2_players
0,2026-03-28,2026,Bengaluru,"M Chinnaswamy Stadium, Bengaluru",Sunrisers Hyderabad,Royal Challengers Bengaluru,Royal Challengers Bengaluru,field,201,9,...,6,JA Duffy,J Srinath,J Madanagopal,UV Gandhe,R Pandit,A Bengeri,20,"TM Head, Abhishek Sharma, Ishan Kishan, Nithis...","JA Duffy, PD Salt, V Kohli, D Padikkal, RM Pat..."
1,2026-03-29,2026,Mumbai,"Wankhede Stadium, Mumbai",Kolkata Knight Riders,Mumbai Indians,Mumbai Indians,field,220,4,...,6,SN Thakur,Shakti Singh,Anish Sahasrabudhe,MV Saidharshan Kumar,Nitin Menon,Vinod Seshan,20,"Kartik Tyagi, AM Rahane, FH Allen, C Green, A ...","JJ Bumrah, RD Rickelton, RG Sharma, SA Yadav, ..."


### Convert Match Records to Text

We convert each row into a natural language description to enable semantic search.

In [15]:
def row_to_text(row):
    text = f"""Match Summary

Date: {row['date']}
Season: IPL {row['season']}

Venue:
{row['venue']}
City: {row['city']}

Teams:
{row['team1']} vs {row['team2']}

Toss:
{row['toss_winner']} won the toss and chose to {row['toss_decision']} first.

First Innings:
{row['team1']} scored {row['team1_runs']} runs for {row['team1_wickets']} wickets.

Second Innings:
{row['team2']} scored {row['team2_runs']} runs for {row['team2_wickets']} wickets.

Match Result:
{row['winner']} won the match.

Winning Margin:

* Runs: {row['win_by_runs']}
* Wickets: {row['win_by_wickets']}

Result Type:
{row['result_type']}

Player of the Match:
{row['player_of_match']}

Match Officials:

* Match Referee: {row['match_referee']}
* Umpire 1: {row['umpire1']}
* Umpire 2: {row['umpire2']}
* TV Umpire: {row['tv_umpire']}
* Reserve Umpire: {row['reserve_umpire']}

Match Facts:

* Match played in {row['city']} during IPL {row['season']}.
* {row['team1']} scored {row['team1_runs']}/{row['team1_wickets']}.
* {row['team2']} scored {row['team2_runs']}/{row['team2_wickets']}.
* {row['winner']} won the match.
* Player of the Match was {row['player_of_match']}.

{row['team1']} Playing XI:
{row['team1_players']}

{row['team2']} Playing XI:
{row['team2_players']}

Search Keywords:
{row['team1']}, {row['team2']}, {row['winner']}, {row['player_of_match']}, {row['city']}, {row['venue']}, IPL {row['season']}"""
    return text

df['match_text'] = df.apply(row_to_text, axis=1)
print("Sample match text representation:")
print(df['match_text'].iloc[0])

Sample match text representation:
Match Summary

Date: 2026-03-28
Season: IPL 2026

Venue:
M Chinnaswamy Stadium, Bengaluru
City: Bengaluru

Teams:
Sunrisers Hyderabad vs Royal Challengers Bengaluru

Toss:
Royal Challengers Bengaluru won the toss and chose to field first.

First Innings:
Sunrisers Hyderabad scored 201 runs for 9 wickets.

Second Innings:
Royal Challengers Bengaluru scored 203 runs for 4 wickets.

Match Result:
Royal Challengers Bengaluru won the match.

Winning Margin:

* Runs: 0
* Wickets: 6

Result Type:
complete

Player of the Match:
JA Duffy

Match Officials:

* Match Referee: J Srinath
* Umpire 1: J Madanagopal
* Umpire 2: UV Gandhe
* TV Umpire: R Pandit
* Reserve Umpire: A Bengeri

Match Facts:

* Match played in Bengaluru during IPL 2026.
* Sunrisers Hyderabad scored 201/9.
* Royal Challengers Bengaluru scored 203/4.
* Royal Challengers Bengaluru won the match.
* Player of the Match was JA Duffy.

Sunrisers Hyderabad Playing XI:
TM Head, Abhishek Sharma, Ishan K

### Initialize ChromaDB Collection

In [16]:
# Initialize persistent ChromaDB client
client = chromadb.PersistentClient(path="./chroma_db")

# Use OpenAI's text-embedding-3-large embedding function
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-large"
)

# Create or get the collection
collection = client.get_or_create_collection(
    name="ipl_matches_2026_openai",
    embedding_function=openai_ef
)
print(f"Collection '{collection.name}' is ready.")

Collection 'ipl_matches_2026_openai' is ready.


### Ingest Match Data into ChromaDB

In [17]:
documents = []
metadatas = []
ids = []

for index, row in df.iterrows():
    # Generate unique ID for each match
    match_id = f"ipl_2026_match_{index}"
    
    # Create metadata dict, replacing NaN values with string representations
    metadata = {
        "date": str(row['date']),
        "season": int(row['season']),
        "city": str(row['city']),
        "venue": str(row['venue']),
        "team1": str(row['team1']),
        "team2": str(row['team2']),
        "winner": str(row['winner']),
        "player_of_match": str(row['player_of_match'])
    }
    
    documents.append(row['match_text'])
    metadatas.append(metadata)
    ids.append(match_id)

# Add data in batches to ChromaDB
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)
print(f"Successfully loaded {len(documents)} matches into ChromaDB!")

Successfully loaded 74 matches into ChromaDB!


### Perform Semantic Search Queries (RAG Retrieval)

In [26]:
def query_matches(query_text, n_results=3):
    results = collection.query(
        query_texts=[query_text],
        n_results=n_results
    )
    
    for i in range(len(results['ids'][0])):
        match_id = results['ids'][0][i]
        document = results['documents'][0][i]
        metadata = results['metadatas'][0][i]
        distance = results['distances'][0][i]
        
        print(f"Result {i+1} (Score/Distance: {distance:.4f}):")
        print(f"ID: {match_id}")
        # print(f"Match Details: {document}")
        print("-" * 80)

# Test query
query_matches("GIVE THE MATCH WHRE WE HAVE MAX SCORE")

Result 1 (Score/Distance: 0.5886):
ID: ipl_2026_match_47
--------------------------------------------------------------------------------
Result 2 (Score/Distance: 0.5904):
ID: ipl_2026_match_65
--------------------------------------------------------------------------------
Result 3 (Score/Distance: 0.5911):
ID: ipl_2026_match_9
--------------------------------------------------------------------------------


### Test other sample queries

In [20]:
query_matches("MI VS CSK", n_results=5)

Result 1 (Score/Distance: 0.3654):
ID: ipl_2026_match_43
Match Details: Match Summary

Date: 2026-05-02
Season: IPL 2026

Venue:
MA Chidambaram Stadium, Chepauk, Chennai
City: Chennai

Teams:
Mumbai Indians vs Chennai Super Kings

Toss:
Mumbai Indians won the toss and chose to bat first.

First Innings:
Mumbai Indians scored 159 runs for 7 wickets.

Second Innings:
Chennai Super Kings scored 160 runs for 2 wickets.

Match Result:
Chennai Super Kings won the match.

Winning Margin:

* Runs: 0
* Wickets: 8

Result Type:
complete

Player of the Match:
RD Gaikwad

Match Officials:

* Match Referee: Amit Sharma
* Umpire 1: KM Gandhi
* Umpire 2: Nitin Menon
* TV Umpire: Vinod Seshan
* Reserve Umpire: TM Srivastava

Match Facts:

* Match played in Chennai during IPL 2026.
* Mumbai Indians scored 159/7.
* Chennai Super Kings scored 160/2.
* Chennai Super Kings won the match.
* Player of the Match was RD Gaikwad.

Mumbai Indians Playing XI:
Raghu Sharma, WG Jacks, RD Rickelton, Naman Dhir, SA Y

### Vector Space Visualization (t-SNE & Plotly)

To visualize how matches group semantically in vector space, we can retrieve the embeddings from ChromaDB, project them using t-SNE (t-distributed Stochastic Neighbor Embedding), and display them interactively using Plotly.

In [21]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']

# Color map based on actual IPL team branding
team_colors = {
    "Royal Challengers Bengaluru": "#CC2336",  # RCB Red
    "Mumbai Indians": "#004BA0",               # MI Blue
    "Rajasthan Royals": "#EA1B85",             # RR Pink
    "Gujarat Titans": "#1B2133",               # GT Dark Navy
    "Punjab Kings": "#DD1F26",                 # PBKS Red
    "Delhi Capitals": "#134980",               # DC Blue
    "Kolkata Knight Riders": "#3A225D",        # KKR Purple
    "Chennai Super Kings": "#FFFF00",          # CSK Yellow
    "Lucknow Super Giants": "#0057B8",         # LSG Blue
    "Sunrisers Hyderabad": "#FF822A",          # SRH Orange
    "None": "#808080"                          # Grey for No Result / Tie
}

winners = [m.get('winner', 'None') for m in metadatas]
colors = [team_colors.get(w, "#A0A0A0") for w in winners]
print(f"Retrieved {len(vectors)} vectors with {len(vectors[0])} dimensions each.")

Retrieved 74 vectors with 3072 dimensions each.


#### 2D Vector Projection

Hover over the points in the plot below to explore details about each match!

In [22]:
# Reduce dimension to 2D using t-SNE
# Using a low perplexity since we have 74 matches
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=15)
reduced_vectors_2d = tsne_2d.fit_transform(vectors)

# Plot the 2D visualization in Dark Theme
fig_2d = go.Figure(data=[go.Scatter(
    x=reduced_vectors_2d[:, 0],
    y=reduced_vectors_2d[:, 1],
    mode='markers',
    marker=dict(
        size=10,
        color=colors,
        opacity=0.85,
        line=dict(width=1, color='rgba(255,255,255,0.3)')
    ),
    text=[
        f"<b>Match:</b> {m.get('team1')} vs {m.get('team2')}<br>"
        f"<b>Winner:</b> {m.get('winner')}<br>"
        f"<b>Date:</b> {m.get('date')}<br>"
        f"<b>Venue:</b> {m.get('venue')}<br>"
        f"<b>POTM:</b> {m.get('player_of_match')}"
        for m in metadatas
    ],
    hoverinfo='text'
)])

fig_2d.update_layout(
    title='2D t-SNE IPL 2026 Match Embeddings Visualization',
    xaxis_title='t-SNE Dimension 1',
    yaxis_title='t-SNE Dimension 2',
    width=900,
    height=600,
    template="plotly_dark"
)

fig_2d.show()

#### 3D Vector Projection

Rotate and zoom to inspect the relative cluster distances.

In [23]:
# Reduce dimension to 3D using t-SNE
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=15)
reduced_vectors_3d = tsne_3d.fit_transform(vectors)

# Plot the 3D visualization in Dark Theme
fig_3d = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors_3d[:, 0],
    y=reduced_vectors_3d[:, 1],
    z=reduced_vectors_3d[:, 2],
    mode='markers',
    marker=dict(
        size=6,
        color=colors,
        opacity=0.85,
        line=dict(width=1, color='rgba(255,255,255,0.3)')
    ),
    text=[
        f"<b>Match:</b> {m.get('team1')} vs {m.get('team2')}<br>"
        f"<b>Winner:</b> {m.get('winner')}<br>"
        f"<b>Date:</b> {m.get('date')}<br>"
        f"<b>Venue:</b> {m.get('venue')}<br>"
        f"<b>POTM:</b> {m.get('player_of_match')}"
        for m in metadatas
    ],
    hoverinfo='text'
)])

fig_3d.update_layout(
    title='3D t-SNE IPL 2026 Match Embeddings Visualization',
    scene=dict(
        xaxis_title='Dim 1',
        yaxis_title='Dim 2',
        zaxis_title='Dim 3'
    ),
    width=900,
    height=750,
    template="plotly_dark"
)

fig_3d.show()